In [1]:
import json
from pathlib import Path

import numpy as np
from PIL import Image
import torch
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoProcessor, CLIPModel

DATA_ROOT = Path("/content/drive/MyDrive/Facebook Hateful Meme Dataset/data")

In [2]:
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
def resolve_annotations_dir():
    required = [DATA_ROOT / "train.jsonl", DATA_ROOT / "dev.jsonl", DATA_ROOT / "test.jsonl"]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "These required split files are missing from /content/drive/MyDrive/Facebook Hateful Meme Dataset/data:\n"
            + "\n".join(missing)
        )
    return DATA_ROOT


class ModerationDataset(Dataset):
    def __init__(self, split, annotations_dir, image_base_dir, transform=None, missing_label_value=-1):
        self.split = split
        self.annotations_dir = Path(annotations_dir)
        self.image_base_dir = Path(image_base_dir)
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])
        self.missing_label_value = missing_label_value
        self.annotations_file = self.annotations_dir / f"{split}.jsonl"
        self.image_root = self.image_base_dir / split
        self.records = []

        if not self.annotations_file.exists():
            raise FileNotFoundError(f"Split file not found for {split}: {self.annotations_file}")
        if not self.image_root.exists():
            raise FileNotFoundError(f"Image folder not found for {split}: {self.image_root}")

        with self.annotations_file.open("r", encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue

                row = json.loads(line)
                image_name = Path(row["img"]).name
                self.records.append(
                    {
                        "image_path": self.image_root / image_name,
                        "caption_text": row["text"],
                        "label": int(row["label"]) if "label" in row else self.missing_label_value,
                    }
                )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        image_path = row["image_path"]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        if isinstance(image, torch.Tensor):
            image = image.clone()
        label = torch.tensor(row["label"], dtype=torch.long)
        return {
            "image": image,
            "image_path": str(image_path),
            "caption_text": row["caption_text"],
            "label": label,
        }


annotations_dir = resolve_annotations_dir()
image_base_dir = DATA_ROOT

train_dataset = ModerationDataset(
    split="train",
    annotations_dir=annotations_dir,
    image_base_dir=image_base_dir,
)
dev_dataset = ModerationDataset(
    split="dev",
    annotations_dir=annotations_dir,
    image_base_dir=image_base_dir,
)
test_dataset = ModerationDataset(
    split="test",
    annotations_dir=annotations_dir,
    image_base_dir=image_base_dir,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

print(f"Using annotations from: {annotations_dir}")
print(f"Number of training samples: {len(train_dataset)}")
first_batch = next(iter(train_loader))
print(first_batch["image"].shape)
print(first_batch["image_path"][:2])
print(first_batch["caption_text"][:2])
print(first_batch["label"][:2])


Using annotations from: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data
Number of training samples: 8500
torch.Size([32, 3, 224, 224])
['/content/drive/MyDrive/Facebook Hateful Meme Dataset/data/train/12704.png', '/content/drive/MyDrive/Facebook Hateful Meme Dataset/data/train/65189.png']
['i cannot comment on your mother because cows are sacred in my country', 'i just sharted ... my protein shake']
tensor([1, 0])


In [9]:
def resolve_annotations_dir_for_clip():
    required = [DATA_ROOT / "train.jsonl", DATA_ROOT / "dev.jsonl", DATA_ROOT / "test.jsonl"]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "These required split files are missing from /content/drive/MyDrive/Facebook Hateful Meme Dataset/data:\n"
            + "\n".join(missing)
        )
    return DATA_ROOT


class CLIPModerationDataset(Dataset):
    def __init__(self, split, annotations_dir, image_base_dir, missing_label_value=-1):
        self.split = split
        self.annotations_file = Path(annotations_dir) / f"{split}.jsonl"
        self.image_root = Path(image_base_dir) / split
        self.missing_label_value = missing_label_value
        self.records = []

        if not self.annotations_file.exists():
            raise FileNotFoundError(f"Split file not found for {split}: {self.annotations_file}")
        if not self.image_root.exists():
            raise FileNotFoundError(f"Image folder not found for {split}: {self.image_root}")

        with self.annotations_file.open("r", encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue

                row = json.loads(line)
                image_name = Path(row["img"]).name
                self.records.append(
                    {
                        "image_path": self.image_root / image_name,
                        "caption_text": row["text"],
                        "label": int(row["label"]) if "label" in row else missing_label_value,
                    }
                )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        return {
            "image": image,
            "caption_text": row["caption_text"],
            "label": row["label"],
            "image_path": str(row["image_path"]),
        }


def clip_collate_fn(batch):
    return {
        "images": [item["image"] for item in batch],
        "caption_text": [item["caption_text"] for item in batch],
        "label": torch.tensor([item["label"] for item in batch], dtype=torch.long),
        "image_path": [item["image_path"] for item in batch],
    }


device = "cuda" if torch.cuda.is_available() else None
if device is None:
    raise RuntimeError("CUDA is not available. Switch the Colab runtime to GPU before extracting CLIP embeddings.")

model_name = "openai/clip-vit-base-patch32"
processor = AutoProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)

for parameter in model.parameters():
    parameter.requires_grad = False

model.eval()
model.to("cuda")

annotations_dir = resolve_annotations_dir_for_clip()
image_base_dir = DATA_ROOT
output_dir = image_base_dir / "clip_embeddings"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Using annotations from: {annotations_dir}")
print(f"Saving CLIP embeddings to: {output_dir}")


def extract_and_save_split(input_split, output_prefix, batch_size=64):
    dataset = CLIPModerationDataset(
        split=input_split,
        annotations_dir=annotations_dir,
        image_base_dir=image_base_dir,
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=clip_collate_fn,
    )

    image_embeddings = []
    text_embeddings = []
    labels = []

    for batch in tqdm(loader, desc=f"Extracting {output_prefix}"):
        inputs = processor(
            text=batch["caption_text"],
            images=batch["images"],
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        pixel_values = inputs["pixel_values"].to(device)
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True,
            )
            batch_image_embeddings = outputs.image_embeds
            batch_text_embeddings = outputs.text_embeds

        batch_image_embeddings = torch.nn.functional.normalize(batch_image_embeddings, dim=-1)
        batch_text_embeddings = torch.nn.functional.normalize(batch_text_embeddings, dim=-1)

        image_embeddings.append(batch_image_embeddings.cpu().numpy().astype(np.float32, copy=False))
        text_embeddings.append(batch_text_embeddings.cpu().numpy().astype(np.float32, copy=False))
        labels.append(batch["label"].cpu().numpy().astype(np.int64, copy=False))

    image_matrix = np.vstack(image_embeddings)
    text_matrix = np.vstack(text_embeddings)
    label_vector = np.concatenate(labels)

    np.save(output_dir / f"{output_prefix}_images.npy", image_matrix)
    np.save(output_dir / f"{output_prefix}_text.npy", text_matrix)
    np.save(output_dir / f"{output_prefix}_labels.npy", label_vector)

    print(f"Saved {output_prefix} image embeddings: {image_matrix.shape}")
    print(f"Saved {output_prefix} text embeddings: {text_matrix.shape}")
    print(f"Saved {output_prefix} labels: {label_vector.shape}")

    return image_matrix, text_matrix, label_vector


train_images, train_text, train_labels = extract_and_save_split("train", "train")
val_images, val_text, val_labels = extract_and_save_split("dev", "val")
test_images, test_text, test_labels = extract_and_save_split("test", "test")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using annotations from: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data
Saving CLIP embeddings to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/clip_embeddings


Extracting train:   0%|          | 0/133 [00:00<?, ?it/s]

Saved train image embeddings: (8500, 512)
Saved train text embeddings: (8500, 512)
Saved train labels: (8500,)


Extracting val:   0%|          | 0/8 [00:00<?, ?it/s]

Saved val image embeddings: (500, 512)
Saved val text embeddings: (500, 512)
Saved val labels: (500,)


Extracting test:   0%|          | 0/16 [00:00<?, ?it/s]

Saved test image embeddings: (1000, 512)
Saved test text embeddings: (1000, 512)
Saved test labels: (1000,)


In [4]:
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import DataLoader, Dataset

class CrossModalEmbeddingDataset(Dataset):
    def __init__(self, image_npy_path, text_npy_path, label_npy_path):
        self.image_npy_path = Path(image_npy_path)
        self.text_npy_path = Path(text_npy_path)
        self.label_npy_path = Path(label_npy_path)

        for path in [self.image_npy_path, self.text_npy_path, self.label_npy_path]:
            if not path.exists():
                raise FileNotFoundError(f"Embedding file not found: {path}")

        self.image_embeddings = np.load(self.image_npy_path, mmap_mode="r")
        self.text_embeddings = np.load(self.text_npy_path, mmap_mode="r")
        self.labels = np.load(self.label_npy_path, mmap_mode="r")

        if len(self.image_embeddings) != len(self.text_embeddings) or len(self.image_embeddings) != len(self.labels):
            raise ValueError("Image, text, and label arrays must have the same number of rows.")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image_vector = np.asarray(self.image_embeddings[idx], dtype=np.float32)
        text_vector = np.asarray(self.text_embeddings[idx], dtype=np.float32)
        label = np.int64(self.labels[idx])

        return {
            "image_features": torch.from_numpy(image_vector.copy()),
            "text_features": torch.from_numpy(text_vector.copy()),
            "label": torch.tensor(label, dtype=torch.long),
        }


LOCAL_EMBEDDING_ROOT = DATA_ROOT / "clip_embeddings"

train_feature_dataset = CrossModalEmbeddingDataset(
    image_npy_path=LOCAL_EMBEDDING_ROOT / "train_images.npy",
    text_npy_path=LOCAL_EMBEDDING_ROOT / "train_text.npy",
    label_npy_path=LOCAL_EMBEDDING_ROOT / "train_labels.npy",
)

train_feature_loader = DataLoader(
    train_feature_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
)

sample_batch = next(iter(train_feature_loader))
print(f"Train rows: {len(train_feature_dataset)}")
print(f"Image feature batch shape: {sample_batch['image_features'].shape}")
print(f"Text feature batch shape: {sample_batch['text_features'].shape}")
print(f"Label batch shape: {sample_batch['label'].shape}")


Train rows: 8500
Image feature batch shape: torch.Size([64, 512])
Text feature batch shape: torch.Size([64, 512])
Label batch shape: torch.Size([64])


In [5]:
val_feature_dataset = CrossModalEmbeddingDataset(
    image_npy_path=LOCAL_EMBEDDING_ROOT / "val_images.npy",
    text_npy_path=LOCAL_EMBEDDING_ROOT / "val_text.npy",
    label_npy_path=LOCAL_EMBEDDING_ROOT / "val_labels.npy",
)

test_feature_dataset = CrossModalEmbeddingDataset(
    image_npy_path=LOCAL_EMBEDDING_ROOT / "test_images.npy",
    text_npy_path=LOCAL_EMBEDDING_ROOT / "test_text.npy",
    label_npy_path=LOCAL_EMBEDDING_ROOT / "test_labels.npy",
)

train_feature_loader = DataLoader(
    train_feature_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
)

val_feature_loader = DataLoader(
    val_feature_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
)

test_feature_loader = DataLoader(
    test_feature_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
)

print(f"Train rows: {len(train_feature_dataset)}")
print(f"Val rows: {len(val_feature_dataset)}")
print(f"Test rows: {len(test_feature_dataset)}")

train_feature_batch = next(iter(train_feature_loader))
print(f"Train image batch shape: {train_feature_batch['image_features'].shape}")
print(f"Train text batch shape: {train_feature_batch['text_features'].shape}")
print(f"Train label batch shape: {train_feature_batch['label'].shape}")


Train rows: 8500
Val rows: 500
Test rows: 1000
Train image batch shape: torch.Size([64, 512])
Train text batch shape: torch.Size([64, 512])
Train label batch shape: torch.Size([64])


In [6]:
import psutil

train_loader = train_feature_loader
process = psutil.Process()
rss_before_mb = process.memory_info().rss / (1024 ** 2)

train_batch = next(iter(train_loader))
image_shape = tuple(train_batch["image_features"].shape)
text_shape = tuple(train_batch["text_features"].shape)
assert image_shape == (64, 512), f"Expected (64, 512), got {image_shape}"
assert text_shape == (64, 512), f"Expected (64, 512), got {text_shape}"
assert torch.isfinite(train_batch["image_features"]).all(), "Found NaN or infinite values in image embeddings."
assert torch.isfinite(train_batch["text_features"]).all(), "Found NaN or infinite values in text embeddings."

rss_after_mb = process.memory_info().rss / (1024 ** 2)

print(f"Image feature batch shape: {image_shape}")
print(f"Text feature batch shape: {text_shape}")
print(f"All image values finite: {bool(torch.isfinite(train_batch['image_features']).all())}")
print(f"All text values finite: {bool(torch.isfinite(train_batch['text_features']).all())}")
print(f"Process RAM before batch: {rss_before_mb:.2f} MB")
print(f"Process RAM after batch: {rss_after_mb:.2f} MB")
print(f"Process RAM delta: {rss_after_mb - rss_before_mb:.2f} MB")
print(train_batch["image_features"])
print(train_batch["text_features"])


Image feature batch shape: (64, 512)
Text feature batch shape: (64, 512)
All image values finite: True
All text values finite: True
Process RAM before batch: 962.25 MB
Process RAM after batch: 963.80 MB
Process RAM delta: 1.55 MB
tensor([[-0.0086, -0.0269,  0.0037,  ...,  0.0809,  0.0059,  0.0710],
        [-0.0384,  0.0138, -0.0207,  ...,  0.0415, -0.0246,  0.0071],
        [-0.0647, -0.0123, -0.0304,  ..., -0.0194, -0.0010, -0.0073],
        ...,
        [-0.0215,  0.0019, -0.0429,  ...,  0.0564, -0.0020,  0.0398],
        [-0.0096,  0.0433, -0.0108,  ..., -0.0196,  0.0208, -0.0367],
        [-0.0324,  0.0057,  0.0098,  ...,  0.0176,  0.0068, -0.0109]])
tensor([[ 3.2939e-03, -2.5510e-02,  3.0974e-02,  ...,  3.6030e-03,
         -7.1731e-03,  1.9542e-02],
        [-4.8052e-03,  2.0015e-02, -2.1295e-02,  ..., -6.7968e-02,
          2.5588e-02, -9.9787e-03],
        [-2.3067e-03,  1.0486e-02, -2.1073e-03,  ...,  2.4187e-02,
         -1.7188e-02, -3.9776e-03],
        ...,
        [-1.11

In [7]:
import torch.nn as nn


class AdvancedMultimodalClassifier(nn.Module):
    def __init__(self, embedding_dim=512, hidden_dim=512, num_heads=8, dropout=0.3):
        super().__init__()
        self.image_projection = nn.Linear(embedding_dim, hidden_dim)
        self.text_projection = nn.Linear(embedding_dim, hidden_dim)
        self.image_to_text_attention = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.text_to_image_attention = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_image = nn.LayerNorm(hidden_dim)
        self.norm_text = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.mish = nn.Mish()
        self.fusion_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Mish(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, image_features, text_features):
        image_token = self.image_projection(image_features).unsqueeze(1)
        text_token = self.text_projection(text_features).unsqueeze(1)

        attended_image, _ = self.image_to_text_attention(image_token, text_token, text_token)
        attended_text, _ = self.text_to_image_attention(text_token, image_token, image_token)

        image_token = self.mish(self.norm_image(image_token + self.dropout(attended_image)))
        text_token = self.mish(self.norm_text(text_token + self.dropout(attended_text)))

        fused = torch.cat([image_token.squeeze(1), text_token.squeeze(1)], dim=-1)
        return self.fusion_mlp(fused)


mlp_model = AdvancedMultimodalClassifier()
forward_image_batch = train_feature_batch["image_features"] if "train_feature_batch" in globals() else next(iter(train_feature_loader))["image_features"]
forward_text_batch = train_feature_batch["text_features"] if "train_feature_batch" in globals() else next(iter(train_feature_loader))["text_features"]
forward_image_batch = forward_image_batch.to(dtype=torch.float32)
forward_text_batch = forward_text_batch.to(dtype=torch.float32)
forward_test_logits = mlp_model(forward_image_batch, forward_text_batch)
assert tuple(forward_test_logits.shape) == (forward_image_batch.shape[0], 1), (
    f"Expected {(forward_image_batch.shape[0], 1)}, got {tuple(forward_test_logits.shape)}"
)
print(mlp_model)
print(f"Forward image batch shape: {forward_image_batch.shape}")
print(f"Forward text batch shape: {forward_text_batch.shape}")
print(f"Forward logits shape: {forward_test_logits.shape}")


AdvancedMultimodalClassifier(
  (image_projection): Linear(in_features=512, out_features=512, bias=True)
  (text_projection): Linear(in_features=512, out_features=512, bias=True)
  (image_to_text_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
  )
  (text_to_image_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
  )
  (norm_image): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (norm_text): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (mish): Mish()
  (fusion_mlp): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): Mish()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=1, bias=True)
  )
)
Forward image batch shape: torch.Size([64, 512])
Forward text batch shape: torch.Size([64, 512])
Forward logits shap

In [8]:
train_labels_path = LOCAL_EMBEDDING_ROOT / "train_labels.npy"
train_labels = np.load(train_labels_path)

num_safe = int((train_labels == 0).sum())
num_harmful = int((train_labels == 1).sum())

if num_harmful == 0:
    raise ValueError("No harmful posts found in train_labels.npy, cannot compute pos_weight.")

pos_weight = num_safe / num_harmful

print(f"Loaded labels from: {train_labels_path}")
print(f"Number of safe posts (class 0): {num_safe}")
print(f"Number of harmful posts (class 1): {num_harmful}")
print(f"pos_weight: {pos_weight:.6f}")


Loaded labels from: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/clip_embeddings/train_labels.npy
Number of safe posts (class 0): 5450
Number of harmful posts (class 1): 3050
pos_weight: 1.786885


In [9]:
import torch.nn.functional as F


class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.40, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        targets = targets.to(dtype=logits.dtype)
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probabilities = torch.sigmoid(logits)
        p_t = probabilities * targets + (1.0 - probabilities) * (1.0 - targets)
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        modulating_factor = (1.0 - p_t).pow(self.gamma)
        loss = alpha_t * modulating_factor * bce_loss

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


device = "cuda" if torch.cuda.is_available() else "cpu"

mlp_model = AdvancedMultimodalClassifier().to(device)
criterion = FocalLossWithLogits(alpha=0.35, gamma=1.5)
optimizer = torch.optim.AdamW(
    mlp_model.parameters(),
    lr=5e-4,
    weight_decay=0.01,
)

print(f"Device: {device}")
print(f"Criterion: {criterion}")
print(f"Optimizer: {optimizer}")


Device: cuda
Criterion: FocalLossWithLogits()
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0.01
)


In [10]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

num_epochs = 15
training_history = []
best_val_f1 = -1.0
best_epoch = None
best_model_state = None
final_validation_probabilities = None
final_validation_targets = None

for epoch in range(1, num_epochs + 1):
    mlp_model.train()
    running_loss = 0.0
    seen_examples = 0

    for batch in train_feature_loader:
        image_features = batch["image_features"].to(device=device, dtype=torch.float32)
        text_features = batch["text_features"].to(device=device, dtype=torch.float32)
        labels = batch["label"].to(device=device, dtype=torch.float32).unsqueeze(1)

        optimizer.zero_grad()
        logits = mlp_model(image_features, text_features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = image_features.size(0)
        running_loss += loss.item() * batch_size
        seen_examples += batch_size

    avg_train_loss = running_loss / seen_examples

    mlp_model.eval()
    val_probabilities = []
    val_targets = []
    val_running_loss = 0.0
    val_seen_examples = 0

    with torch.no_grad():
        for batch in val_feature_loader:
            image_features = batch["image_features"].to(device=device, dtype=torch.float32)
            text_features = batch["text_features"].to(device=device, dtype=torch.float32)
            labels = batch["label"].to(device=device, dtype=torch.float32).unsqueeze(1)

            logits = mlp_model(image_features, text_features)
            val_loss = criterion(logits, labels)
            probabilities = torch.sigmoid(logits)

            batch_size = image_features.size(0)
            val_running_loss += val_loss.item() * batch_size
            val_seen_examples += batch_size

            val_probabilities.extend(probabilities.squeeze(1).cpu().tolist())
            val_targets.extend(labels.squeeze(1).cpu().tolist())

    avg_val_loss = val_running_loss / val_seen_examples
    val_probabilities = np.asarray(val_probabilities, dtype=np.float32)
    val_targets = np.asarray(val_targets, dtype=np.int64)
    final_validation_probabilities = val_probabilities.copy()
    final_validation_targets = val_targets.copy()
    val_predictions = (val_probabilities >= 0.5).astype(np.int64)

    val_accuracy = accuracy_score(val_targets, val_predictions)
    val_precision = precision_score(val_targets, val_predictions, zero_division=0)
    val_recall = recall_score(val_targets, val_predictions, zero_division=0)
    val_f1 = f1_score(val_targets, val_predictions, zero_division=0)

    training_history.append(
        {
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_accuracy": val_accuracy,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_f1": val_f1,
        }
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in mlp_model.state_dict().items()
        }

    print(
        f"Epoch {epoch:02d}/{num_epochs} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"val_loss={avg_val_loss:.4f} | "
        f"val_acc={val_accuracy:.4f} | "
        f"val_precision={val_precision:.4f} | "
        f"val_recall={val_recall:.4f} | "
        f"val_f1={val_f1:.4f}"
    )

print(f"Best validation F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Stored final validation probabilities: {final_validation_probabilities.shape}")
print(f"Stored final validation targets: {final_validation_targets.shape}")


Epoch 01/15 | train_loss=0.0981 | val_loss=0.1470 | val_acc=0.5000 | val_precision=0.5000 | val_recall=0.0040 | val_f1=0.0079
Epoch 02/15 | train_loss=0.0864 | val_loss=0.1085 | val_acc=0.5720 | val_precision=0.6957 | val_recall=0.2560 | val_f1=0.3743
Epoch 03/15 | train_loss=0.0838 | val_loss=0.1107 | val_acc=0.5380 | val_precision=0.6727 | val_recall=0.1480 | val_f1=0.2426
Epoch 04/15 | train_loss=0.0802 | val_loss=0.1155 | val_acc=0.5400 | val_precision=0.7778 | val_recall=0.1120 | val_f1=0.1958
Epoch 05/15 | train_loss=0.0780 | val_loss=0.1385 | val_acc=0.5880 | val_precision=0.7075 | val_recall=0.3000 | val_f1=0.4213
Epoch 06/15 | train_loss=0.0747 | val_loss=0.1393 | val_acc=0.5360 | val_precision=0.7500 | val_recall=0.1080 | val_f1=0.1888
Epoch 07/15 | train_loss=0.0713 | val_loss=0.1405 | val_acc=0.5140 | val_precision=0.7333 | val_recall=0.0440 | val_f1=0.0830
Epoch 08/15 | train_loss=0.0708 | val_loss=0.1414 | val_acc=0.5840 | val_precision=0.6842 | val_recall=0.3120 | val_f1

In [11]:
mlp_model.eval()
final_validation_probabilities = []
final_validation_targets = []

with torch.no_grad():
    for batch in val_feature_loader:
        image_features = batch["image_features"].to(device=device, dtype=torch.float32)
        text_features = batch["text_features"].to(device=device, dtype=torch.float32)
        labels = batch["label"].to(device=device, dtype=torch.float32).unsqueeze(1)

        logits = mlp_model(image_features, text_features)
        probabilities = torch.sigmoid(logits)

        final_validation_probabilities.extend(probabilities.squeeze(1).cpu().tolist())
        final_validation_targets.extend(labels.squeeze(1).cpu().tolist())

final_validation_probabilities = np.asarray(final_validation_probabilities, dtype=np.float32)
final_validation_targets = np.asarray(final_validation_targets, dtype=np.int64)

print(f"Collected validation probabilities: {final_validation_probabilities.shape}")
print(f"Collected validation targets: {final_validation_targets.shape}")


Collected validation probabilities: (500,)
Collected validation targets: (500,)


In [12]:
final_model_path = DATA_ROOT / "mlp_final_model.pt"
best_model_path = DATA_ROOT / "mlp_best_val_model.pt"

torch.save(
    {
        "model_state_dict": mlp_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "pos_weight": pos_weight,
        "training_history": training_history,
        "num_epochs": num_epochs,
        "device": device,
    },
    final_model_path,
)

if best_model_state is None:
    raise RuntimeError("best_model_state is not available. Run the training cell first.")

torch.save(
    {
        "model_state_dict": best_model_state,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "pos_weight": pos_weight,
        "training_history": training_history,
        "num_epochs": num_epochs,
    },
    best_model_path,
)

print(f"Saved final model to: {final_model_path}")
print(f"Saved best validation model to: {best_model_path}")
print(f"Best validation epoch: {best_epoch}")
print(f"Best validation F1: {best_val_f1:.4f}")


Saved final model to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/mlp_final_model.pt
Saved best validation model to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/mlp_best_val_model.pt
Best validation epoch: 14
Best validation F1: 0.4457


In [13]:
from sklearn.metrics import precision_recall_fscore_support

if final_validation_probabilities is None or final_validation_targets is None:
    raise RuntimeError("Run the training cell first so final validation predictions are available.")

def sweep_thresholds(probabilities, targets, start=0.01, stop=0.99, step=0.01):
    results = []
    for threshold in np.arange(start, stop + 1e-9, step):
        predictions = (probabilities >= threshold).astype(np.int64)
        precision, recall, f1, _ = precision_recall_fscore_support(
            targets,
            predictions,
            average="binary",
            zero_division=0,
        )
        results.append(
            {
                "threshold": float(np.round(threshold, 2)),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
            }
        )
    return results

threshold_results = sweep_thresholds(final_validation_probabilities, final_validation_targets)
eligible_thresholds = [row for row in threshold_results if row["recall"] >= 0.95]

if eligible_thresholds:
    best_operating_point = max(
        eligible_thresholds,
        key=lambda row: (row["precision"], -row["threshold"]),
    )
    threshold_selection_mode = "recall_constrained"
else:
    best_operating_point = max(
        threshold_results,
        key=lambda row: (row["recall"], row["precision"], -row["threshold"]),
    )
    threshold_selection_mode = "fallback_highest_recall"
prod_threshold = best_operating_point["threshold"]
baseline_precision = 0.533
precision_delta = best_operating_point["precision"] - baseline_precision
meets_recall_constraint = threshold_selection_mode == "recall_constrained"
beats_precision_target = best_operating_point["precision"] > baseline_precision

print(f"Collected validation probabilities: {final_validation_probabilities.shape}")
print(f"Collected validation targets: {final_validation_targets.shape}")
print(f"Thresholds swept: {len(threshold_results)}")
print(f"Eligible thresholds (recall >= 0.95): {len(eligible_thresholds)}")
print(f"Threshold selection mode: {threshold_selection_mode}")
if threshold_selection_mode == "fallback_highest_recall":
    print("No threshold reached recall >= 0.95; using the highest-recall operating point instead.")
print(f"prod_threshold = {prod_threshold:.2f}")
print(best_operating_point)
print(f"Target precision baseline: {baseline_precision:.3f}")
print(f"New operating precision: {best_operating_point['precision']:.3f}")
print(f"Precision delta vs baseline: {precision_delta:.3f}")
print(f"Meets recall >= 0.95 constraint: {meets_recall_constraint}")
print(f"Improved over 53.3% baseline: {beats_precision_target}")


Collected validation probabilities: (500,)
Collected validation targets: (500,)
Thresholds swept: 99
Eligible thresholds (recall >= 0.95): 3
Threshold selection mode: recall_constrained
prod_threshold = 0.03
{'threshold': 0.03, 'precision': 0.525974025974026, 'recall': 0.972, 'f1': 0.6825842696629213}
Target precision baseline: 0.533
New operating precision: 0.526
Precision delta vs baseline: -0.007
Meets recall >= 0.95 constraint: True
Improved over 53.3% baseline: False


In [14]:
fusion_head_path = DATA_ROOT / "multimodal_fusion_head.pth"
torch.save(mlp_model.state_dict(), fusion_head_path)
print(f"Saved fusion head weights to: {fusion_head_path}")
if 'prod_threshold' in globals():
    print(f"Production threshold: {prod_threshold:.2f}")


Saved fusion head weights to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/multimodal_fusion_head.pth
Production threshold: 0.03


In [15]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

test_threshold = prod_threshold if 'prod_threshold' in globals() else 0.5
checkpoint_path = DATA_ROOT / "mlp_best_val_model.pt"

test_model = AdvancedMultimodalClassifier().to(device)
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    test_model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best validation checkpoint from: {checkpoint_path}")
else:
    test_model.load_state_dict(mlp_model.state_dict())
    print("Best validation checkpoint not found; using current in-memory model weights.")

test_model.eval()
test_probabilities = []
test_targets = []

with torch.no_grad():
    for batch in test_feature_loader:
        image_features = batch["image_features"].to(device=device, dtype=torch.float32)
        text_features = batch["text_features"].to(device=device, dtype=torch.float32)
        labels = batch["label"].to(device=device, dtype=torch.float32).unsqueeze(1)

        logits = test_model(image_features, text_features)
        probabilities = torch.sigmoid(logits)

        test_probabilities.extend(probabilities.squeeze(1).cpu().tolist())
        test_targets.extend(labels.squeeze(1).cpu().tolist())

test_probabilities = np.asarray(test_probabilities, dtype=np.float32)
test_targets = np.asarray(test_targets, dtype=np.int64)
test_predictions = (test_probabilities >= test_threshold).astype(np.int64)
valid_label_mask = np.isin(test_targets, [0, 1])

np.save(DATA_ROOT / "test_probabilities.npy", test_probabilities)
np.save(DATA_ROOT / "test_predictions.npy", test_predictions)

print(f"Test threshold: {test_threshold:.2f}")
print(f"Collected test probabilities: {test_probabilities.shape}")
print(f"Collected test targets: {test_targets.shape}")
print(f"Predicted harmful count: {int(test_predictions.sum())}")
print(f"Predicted safe count: {int((test_predictions == 0).sum())}")
print(f"Saved probabilities to: {DATA_ROOT / 'test_probabilities.npy'}")
print(f"Saved predictions to: {DATA_ROOT / 'test_predictions.npy'}")

if valid_label_mask.all():
    test_accuracy = accuracy_score(test_targets, test_predictions)
    test_precision = precision_score(test_targets, test_predictions, zero_division=0)
    test_recall = recall_score(test_targets, test_predictions, zero_division=0)
    test_f1 = f1_score(test_targets, test_predictions, zero_division=0)
    print(f"test_acc={test_accuracy:.4f}")
    print(f"test_precision={test_precision:.4f}")
    print(f"test_recall={test_recall:.4f}")
    print(f"test_f1={test_f1:.4f}")
else:
    print("Test labels are not available as binary 0/1 targets, so metrics are skipped.")


Loaded best validation checkpoint from: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/mlp_best_val_model.pt
Test threshold: 0.03
Collected test probabilities: (1000,)
Collected test targets: (1000,)
Predicted harmful count: 904
Predicted safe count: 96
Saved probabilities to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/test_probabilities.npy
Saved predictions to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/test_predictions.npy
Test labels are not available as binary 0/1 targets, so metrics are skipped.


In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier


def load_best_stage1_model(checkpoint_path, device):
    stage1_model = AdvancedMultimodalClassifier().to(device)
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=device)
        stage1_model.load_state_dict(checkpoint["model_state_dict"])
        print(f"Loaded Stage 1 checkpoint from: {checkpoint_path}")
    elif "best_model_state" in globals() and best_model_state is not None:
        stage1_model.load_state_dict(best_model_state)
        print("Using in-memory Stage 1 best_model_state.")
    else:
        stage1_model.load_state_dict(mlp_model.state_dict())
        print("Best checkpoint not found; using current in-memory Stage 1 weights.")
    stage1_model.eval()
    return stage1_model


def collect_stage1_outputs(dataset, stage1_model, device, batch_size=256):
    image_array = np.asarray(dataset.image_embeddings, dtype=np.float32)
    text_array = np.asarray(dataset.text_embeddings, dtype=np.float32)
    labels = np.asarray(dataset.labels, dtype=np.int64)
    probabilities = []

    with torch.no_grad():
        for start in range(0, len(labels), batch_size):
            stop = min(start + batch_size, len(labels))
            batch_image = torch.from_numpy(image_array[start:stop]).to(device=device, dtype=torch.float32)
            batch_text = torch.from_numpy(text_array[start:stop]).to(device=device, dtype=torch.float32)
            batch_logits = stage1_model(batch_image, batch_text)
            batch_probabilities = torch.sigmoid(batch_logits).squeeze(1).cpu().numpy()
            probabilities.append(batch_probabilities.astype(np.float32, copy=False))

    stage1_probabilities = np.concatenate(probabilities)
    return {
        "image_features": image_array,
        "text_features": text_array,
        "labels": labels,
        "stage1_probabilities": stage1_probabilities,
    }


stage1_checkpoint_path = DATA_ROOT / "mlp_best_val_model.pt"
stage1_model = load_best_stage1_model(stage1_checkpoint_path, device)
stage1_train_outputs = collect_stage1_outputs(train_feature_dataset, stage1_model, device)
stage1_val_outputs = collect_stage1_outputs(val_feature_dataset, stage1_model, device)

print(f"Collected Stage 1 train probabilities: {stage1_train_outputs['stage1_probabilities'].shape}")
print(f"Collected Stage 1 val probabilities: {stage1_val_outputs['stage1_probabilities'].shape}")


Loaded Stage 1 checkpoint from: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/mlp_best_val_model.pt
Collected Stage 1 train probabilities: (8500,)
Collected Stage 1 val probabilities: (500,)


/tmp/ipykernel_3261/2691164835.py:29: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  batch_image = torch.from_numpy(image_array[start:stop]).to(device=device, dtype=torch.float32)


In [17]:
from sklearn.metrics import precision_recall_fscore_support


def build_stage2_feature_matrix(image_features, text_features, stage1_probabilities):
    interaction = image_features * text_features
    absolute_gap = np.abs(image_features - text_features)
    stage1_column = stage1_probabilities.reshape(-1, 1).astype(np.float32, copy=False)
    return np.hstack([image_features, text_features, interaction, absolute_gap, stage1_column]).astype(np.float32, copy=False)


def choose_screening_threshold(probabilities, labels, min_recall=0.995):
    candidates = []
    for threshold in np.arange(0.01, 0.51, 0.01):
        predictions = (probabilities >= threshold).astype(np.int64)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            zero_division=0,
        )
        candidates.append(
            {
                "threshold": float(np.round(threshold, 2)),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
            }
        )

    eligible = [row for row in candidates if row["recall"] >= min_recall]
    if eligible:
        best = max(eligible, key=lambda row: (row["precision"], -row["threshold"]))
        mode = "recall_constrained"
    else:
        best = max(candidates, key=lambda row: (row["recall"], row["precision"], -row["threshold"]))
        mode = "fallback_highest_recall"
    return best, mode, candidates


screening_choice, screening_mode, screening_candidates = choose_screening_threshold(
    stage1_val_outputs["stage1_probabilities"],
    stage1_val_outputs["labels"],
    min_recall=0.995,
)
stage1_screen_threshold = screening_choice["threshold"]

train_suspicious_mask = (stage1_train_outputs["stage1_probabilities"] >= stage1_screen_threshold) | (stage1_train_outputs["labels"] == 1)
val_suspicious_mask = (stage1_val_outputs["stage1_probabilities"] >= stage1_screen_threshold) | (stage1_val_outputs["labels"] == 1)

X_stage2_train = build_stage2_feature_matrix(
    stage1_train_outputs["image_features"][train_suspicious_mask],
    stage1_train_outputs["text_features"][train_suspicious_mask],
    stage1_train_outputs["stage1_probabilities"][train_suspicious_mask],
)
y_stage2_train = stage1_train_outputs["labels"][train_suspicious_mask]
X_stage2_val = build_stage2_feature_matrix(
    stage1_val_outputs["image_features"][val_suspicious_mask],
    stage1_val_outputs["text_features"][val_suspicious_mask],
    stage1_val_outputs["stage1_probabilities"][val_suspicious_mask],
)
y_stage2_val = stage1_val_outputs["labels"][val_suspicious_mask]

num_stage2_neg = int((y_stage2_train == 0).sum())
num_stage2_pos = int((y_stage2_train == 1).sum())
if num_stage2_pos == 0:
    raise ValueError("Stage 2 training set has no positive examples.")
stage2_pos_weight = num_stage2_neg / max(num_stage2_pos, 1)

stage2_model = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.05,
    max_iter=250,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1e-2,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
)
stage2_sample_weight = np.where(y_stage2_train == 1, stage2_pos_weight, 1.0).astype(np.float32)
stage2_model.fit(X_stage2_train, y_stage2_train, sample_weight=stage2_sample_weight)
stage2_val_probabilities = stage2_model.predict_proba(X_stage2_val)[:, 1].astype(np.float32, copy=False)

print(f"Stage 1 screening threshold: {stage1_screen_threshold:.2f}")
print(f"Stage 1 screening mode: {screening_mode}")
print(f"Stage 2 train rows: {X_stage2_train.shape}")
print(f"Stage 2 val rows: {X_stage2_val.shape}")
print(f"Stage 2 positive weight: {stage2_pos_weight:.4f}")


Stage 1 screening threshold: 0.01
Stage 1 screening mode: recall_constrained
Stage 2 train rows: (8191, 2049)
Stage 2 val rows: (491, 2049)
Stage 2 positive weight: 1.6856


In [18]:
def apply_two_stage_pipeline(stage1_outputs, stage2_model, stage1_screen_threshold, stage2_threshold):
    stage1_probabilities = stage1_outputs["stage1_probabilities"]
    suspicious_mask = stage1_probabilities >= stage1_screen_threshold
    final_probabilities = np.zeros_like(stage1_probabilities, dtype=np.float32)
    final_predictions = np.zeros_like(stage1_outputs["labels"], dtype=np.int64)

    if suspicious_mask.any():
        stage2_features = build_stage2_feature_matrix(
            stage1_outputs["image_features"][suspicious_mask],
            stage1_outputs["text_features"][suspicious_mask],
            stage1_outputs["stage1_probabilities"][suspicious_mask],
        )
        suspicious_probabilities = stage2_model.predict_proba(stage2_features)[:, 1].astype(np.float32, copy=False)
        final_probabilities[suspicious_mask] = suspicious_probabilities
        final_predictions[suspicious_mask] = (suspicious_probabilities >= stage2_threshold).astype(np.int64)

    return final_probabilities, final_predictions, suspicious_mask


stage2_threshold_results = sweep_thresholds(stage2_val_probabilities, y_stage2_val)
stage2_eligible_thresholds = [row for row in stage2_threshold_results if row["recall"] >= 0.95]
if stage2_eligible_thresholds:
    best_stage2_operating_point = max(
        stage2_eligible_thresholds,
        key=lambda row: (row["precision"], -row["threshold"]),
    )
    stage2_threshold_mode = "recall_constrained"
else:
    best_stage2_operating_point = max(
        stage2_threshold_results,
        key=lambda row: (row["recall"], row["precision"], -row["threshold"]),
    )
    stage2_threshold_mode = "fallback_highest_recall"
stage2_threshold = best_stage2_operating_point["threshold"]

two_stage_val_probabilities, two_stage_val_predictions, two_stage_val_mask = apply_two_stage_pipeline(
    stage1_val_outputs,
    stage2_model,
    stage1_screen_threshold,
    stage2_threshold,
)

two_stage_val_accuracy = accuracy_score(stage1_val_outputs["labels"], two_stage_val_predictions)
two_stage_val_precision = precision_score(stage1_val_outputs["labels"], two_stage_val_predictions, zero_division=0)
two_stage_val_recall = recall_score(stage1_val_outputs["labels"], two_stage_val_predictions, zero_division=0)
two_stage_val_f1 = f1_score(stage1_val_outputs["labels"], two_stage_val_predictions, zero_division=0)

print(f"Stage 2 threshold mode: {stage2_threshold_mode}")
print(f"Stage 2 threshold: {stage2_threshold:.2f}")
print(f"Stage 1 suspicious val samples: {int(two_stage_val_mask.sum())} / {len(two_stage_val_mask)}")
print(f"Two-stage val_acc={two_stage_val_accuracy:.4f}")
print(f"Two-stage val_precision={two_stage_val_precision:.4f}")
print(f"Two-stage val_recall={two_stage_val_recall:.4f}")
print(f"Two-stage val_f1={two_stage_val_f1:.4f}")
print(best_stage2_operating_point)


Stage 2 threshold mode: recall_constrained
Stage 2 threshold: 0.01
Stage 1 suspicious val samples: 490 / 500
Two-stage val_acc=0.5160
Two-stage val_precision=0.5082
Two-stage val_recall=0.9960
Two-stage val_f1=0.6730
{'threshold': 0.01, 'precision': 0.5091649694501018, 'recall': 1.0, 'f1': 0.6747638326585695}


In [19]:
import joblib

stage2_model_path = DATA_ROOT / "stage2_precision_model.joblib"
two_stage_config_path = DATA_ROOT / "two_stage_config.npz"
joblib.dump(stage2_model, stage2_model_path)
np.savez(
    two_stage_config_path,
    stage1_screen_threshold=np.array([stage1_screen_threshold], dtype=np.float32),
    stage2_threshold=np.array([stage2_threshold], dtype=np.float32),
)
print(f"Saved Stage 2 model to: {stage2_model_path}")
print(f"Saved two-stage thresholds to: {two_stage_config_path}")


Saved Stage 2 model to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/stage2_precision_model.joblib
Saved two-stage thresholds to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/two_stage_config.npz


In [20]:
stage1_test_outputs = collect_stage1_outputs(test_feature_dataset, stage1_model, device)
two_stage_test_probabilities, two_stage_test_predictions, two_stage_test_mask = apply_two_stage_pipeline(
    stage1_test_outputs,
    stage2_model,
    stage1_screen_threshold,
    stage2_threshold,
)
np.save(DATA_ROOT / "two_stage_test_probabilities.npy", two_stage_test_probabilities)
np.save(DATA_ROOT / "two_stage_test_predictions.npy", two_stage_test_predictions)
print(f"Stage 1 suspicious test samples: {int(two_stage_test_mask.sum())} / {len(two_stage_test_mask)}")
print(f"Predicted harmful count: {int(two_stage_test_predictions.sum())}")
print(f"Predicted safe count: {int((two_stage_test_predictions == 0).sum())}")
print(f"Saved probabilities to: {DATA_ROOT / 'two_stage_test_probabilities.npy'}")
print(f"Saved predictions to: {DATA_ROOT / 'two_stage_test_predictions.npy'}")


Stage 1 suspicious test samples: 971 / 1000
Predicted harmful count: 971
Predicted safe count: 29
Saved probabilities to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/two_stage_test_probabilities.npy
Saved predictions to: /content/drive/MyDrive/Facebook Hateful Meme Dataset/data/two_stage_test_predictions.npy


In [ ]:
import json
import sys
import numpy as np
import torch
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import precision_recall_fscore_support


data_root = DATA_ROOT if "DATA_ROOT" in globals() else Path("/content/drive/MyDrive/Facebook Hateful Meme Dataset/data")
train_jsonl = data_root / "train.jsonl"
val_jsonl = data_root / "dev.jsonl"
test_jsonl = data_root / "test.jsonl"

missing = [str(p) for p in [train_jsonl, val_jsonl, test_jsonl] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing caption split files in DATA_ROOT:\n" + "\n".join(missing))


def read_jsonl_text_and_labels(path, label_key="label"):
    texts = []
    labels = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            texts.append(str(row.get("text", "")))
            labels.append(int(row[label_key]) if label_key in row else -1)
    return texts, np.asarray(labels, dtype=np.int64)


train_texts, train_labels = read_jsonl_text_and_labels(train_jsonl)
val_texts, val_labels = read_jsonl_text_and_labels(val_jsonl)
test_texts, test_labels = read_jsonl_text_and_labels(test_jsonl)

valid_train_mask = np.isin(train_labels, [0, 1])
valid_val_mask = np.isin(val_labels, [0, 1])
train_texts = [t for t, keep in zip(train_texts, valid_train_mask) if keep]
val_texts = [t for t, keep in zip(val_texts, valid_val_mask) if keep]
train_labels = train_labels[valid_train_mask]
val_labels = val_labels[valid_val_mask]


try:
    from transformers import AutoModel, AutoTokenizer
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    from transformers import AutoModel, AutoTokenizer


device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sentence-transformers/all-MiniLM-L6-v2"
max_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name).to(device)
encoder.eval()


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(dtype=last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp_min(1e-6)
    return summed / counts


def encode_texts(texts, batch_size=256):
    vectors = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = encoder(**inputs)
            pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled, dim=-1)
            vectors.append(pooled.cpu().numpy().astype(np.float32, copy=False))
    return np.vstack(vectors)


embed_dir = data_root / "caption_only_embeddings"
embed_dir.mkdir(parents=True, exist_ok=True)
train_embed_path = embed_dir / "train_minilm.npy"
val_embed_path = embed_dir / "val_minilm.npy"

if train_embed_path.exists():
    X_train = np.load(train_embed_path, mmap_mode="r")
    print(f"Loaded cached train caption embeddings: {train_embed_path} {X_train.shape}")
else:
    X_train = encode_texts(train_texts)
    np.save(train_embed_path, X_train)
    print(f"Saved train caption embeddings: {train_embed_path} {X_train.shape}")

if val_embed_path.exists():
    X_val = np.load(val_embed_path, mmap_mode="r")
    print(f"Loaded cached val caption embeddings: {val_embed_path} {X_val.shape}")
else:
    X_val = encode_texts(val_texts)
    np.save(val_embed_path, X_val)
    print(f"Saved val caption embeddings: {val_embed_path} {X_val.shape}")


num_safe = int((train_labels == 0).sum())
num_harmful = int((train_labels == 1).sum())
if num_harmful == 0:
    raise ValueError("No harmful posts found in train.jsonl, cannot compute class weights.")
pos_weight = num_safe / num_harmful

caption_text_model = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=2000,
    class_weight={0: 1.0, 1: float(pos_weight)},
)
caption_text_model.fit(np.asarray(X_train), train_labels)
val_probabilities = caption_text_model.predict_proba(np.asarray(X_val))[:, 1].astype(np.float32, copy=False)
val_predictions = (val_probabilities >= 0.5).astype(np.int64)

val_acc = accuracy_score(val_labels, val_predictions)
val_precision = precision_score(val_labels, val_predictions, zero_division=0)
val_recall = recall_score(val_labels, val_predictions, zero_division=0)
val_f1 = f1_score(val_labels, val_predictions, zero_division=0)
print(f"Caption-only (MiniLM + LogisticRegression) val_acc={val_acc:.4f} val_precision={val_precision:.4f} val_recall={val_recall:.4f} val_f1={val_f1:.4f}")


def sweep_thresholds(probabilities, targets, start=0.01, stop=0.99, step=0.01):
    results = []
    for threshold in np.arange(start, stop + 1e-9, step):
        predictions = (probabilities >= threshold).astype(np.int64)
        precision, recall, f1, _ = precision_recall_fscore_support(
            targets,
            predictions,
            average="binary",
            zero_division=0,
        )
        results.append(
            {
                "threshold": float(np.round(threshold, 2)),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
            }
        )
    return results


threshold_results = sweep_thresholds(val_probabilities, val_labels)
eligible = [row for row in threshold_results if row["recall"] >= 0.95]
if eligible:
    best_point = max(eligible, key=lambda row: (row["precision"], -row["threshold"]))
    threshold_mode = "recall_constrained"
else:
    best_point = max(threshold_results, key=lambda row: (row["recall"], row["precision"], -row["threshold"]))
    threshold_mode = "fallback_highest_recall"
caption_prod_threshold = best_point["threshold"]


import joblib
caption_model_path = data_root / "caption_only_minilm_logreg.joblib"
caption_config_path = data_root / "caption_only_config.npz"

joblib.dump(
    {
        "model": caption_text_model,
        "model_name": model_name,
        "max_length": max_length,
    },
    caption_model_path,
)

np.save(data_root / "caption_only_val_probabilities.npy", val_probabilities)
np.save(data_root / "caption_only_val_targets.npy", val_labels)
np.savez(
    caption_config_path,
    caption_prod_threshold=np.array([caption_prod_threshold], dtype=np.float32),
    model_name=np.array([model_name]),
    max_length=np.array([max_length], dtype=np.int32),
)

print(f"Caption-only threshold mode: {threshold_mode}")
print(f"caption_prod_threshold = {caption_prod_threshold:.2f}")
print(best_point)
print(f"Saved caption-only model to: {caption_model_path}")
print(f"Saved caption-only config to: {caption_config_path}")


In [3]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import torch
from sklearn.metrics import precision_recall_fscore_support

try:
    from transformers import AutoModel, AutoTokenizer
except Exception:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    from transformers import AutoModel, AutoTokenizer


if "AdvancedMultimodalClassifier" not in globals():
    raise RuntimeError(
        "AdvancedMultimodalClassifier is not defined. Run the multimodal model definition cell before the ensemble cell."
    )

data_root = DATA_ROOT if "DATA_ROOT" in globals() else Path("/content/drive/MyDrive/Facebook Hateful Meme Dataset/data")
embedding_root = data_root / "clip_embeddings"
caption_model_path = data_root / "caption_only_minilm_logreg.joblib"
caption_config_path = data_root / "caption_only_config.npz"
caption_val_prob_path = data_root / "caption_only_val_probabilities.npy"
caption_val_target_path = data_root / "caption_only_val_targets.npy"
test_jsonl_path = data_root / "test.jsonl"
output_jsonl_path = data_root / "ensemble_test_predictions.jsonl"
output_npy_path = data_root / "ensemble_test_probabilities.npy"
ensemble_config_path = data_root / "ensemble_config.npz"
multimodal_checkpoint_path = data_root / "mlp_best_val_model.pt"
multimodal_fallback_path = data_root / "multimodal_fusion_head.pth"

required_paths = [caption_model_path, caption_config_path, test_jsonl_path]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing ensemble artifacts:\n" + "\n".join(missing))


def read_test_records(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(dtype=last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp_min(1e-6)
    return summed / counts


def encode_texts(texts, model_name, max_length, device, cache_path=None, batch_size=256):
    if cache_path is not None and cache_path.exists():
        return np.load(cache_path, mmap_mode="r")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    encoder = AutoModel.from_pretrained(model_name).to(device)
    encoder.eval()
    vectors = []

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = encoder(**inputs)
            pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled, dim=-1)
            vectors.append(pooled.cpu().numpy().astype(np.float32, copy=False))

    matrix = np.vstack(vectors)
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(cache_path, matrix)
    return matrix


def load_split_embeddings(split_name):
    image_path = embedding_root / f"{split_name}_images.npy"
    text_path = embedding_root / f"{split_name}_text.npy"
    label_path = embedding_root / f"{split_name}_labels.npy"

    missing = [str(path) for path in [image_path, text_path, label_path] if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing multimodal embedding files:\n" + "\n".join(missing))

    image_features = np.load(image_path, mmap_mode="r")
    text_features = np.load(text_path, mmap_mode="r")
    labels = np.load(label_path, mmap_mode="r")
    if len(image_features) != len(text_features) or len(image_features) != len(labels):
        raise ValueError(f"Split {split_name} has mismatched image/text/label row counts.")
    return image_features, text_features, np.asarray(labels, dtype=np.int64)


def load_multimodal_model(device):
    model = AdvancedMultimodalClassifier().to(device)
    checkpoint_source = None

    if multimodal_checkpoint_path.exists():
        checkpoint = torch.load(multimodal_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        checkpoint_source = str(multimodal_checkpoint_path)
    elif multimodal_fallback_path.exists():
        state_dict = torch.load(multimodal_fallback_path, map_location=device)
        if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        model.load_state_dict(state_dict)
        checkpoint_source = str(multimodal_fallback_path)
    elif "best_model_state" in globals() and best_model_state is not None:
        model.load_state_dict(best_model_state)
        checkpoint_source = "in_memory_best_model_state"
    else:
        raise FileNotFoundError(
            "No multimodal checkpoint found. Expected mlp_best_val_model.pt or multimodal_fusion_head.pth in DATA_ROOT."
        )

    model.eval()
    return model, checkpoint_source


def collect_multimodal_probabilities(model, image_features, text_features, device, batch_size=256):
    probabilities = []
    with torch.no_grad():
        for start in range(0, len(image_features), batch_size):
            stop = min(start + batch_size, len(image_features))
            batch_image = torch.from_numpy(np.asarray(image_features[start:stop], dtype=np.float32)).to(device)
            batch_text = torch.from_numpy(np.asarray(text_features[start:stop], dtype=np.float32)).to(device)
            logits = model(batch_image, batch_text)
            batch_probabilities = torch.sigmoid(logits).squeeze(1).cpu().numpy()
            probabilities.append(batch_probabilities.astype(np.float32, copy=False))
    return np.concatenate(probabilities)


def sweep_thresholds(probabilities, targets, start=0.01, stop=0.99, step=0.01):
    results = []
    for threshold in np.arange(start, stop + 1e-9, step):
        predictions = (probabilities >= threshold).astype(np.int64)
        precision, recall, f1, _ = precision_recall_fscore_support(
            targets,
            predictions,
            average="binary",
            zero_division=0,
        )
        results.append(
            {
                "threshold": float(np.round(threshold, 2)),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
            }
        )
    return results


def select_best_operating_point(probabilities, targets, min_recall=0.95):
    threshold_results = sweep_thresholds(probabilities, targets)
    eligible = [row for row in threshold_results if row["recall"] >= min_recall]
    if eligible:
        best_point = max(eligible, key=lambda row: (row["precision"], -row["threshold"]))
        mode = "recall_constrained"
    else:
        best_point = max(threshold_results, key=lambda row: (row["recall"], row["precision"], -row["threshold"]))
        mode = "fallback_highest_recall"
    return best_point, mode


def choose_ensemble_weight(multimodal_val_probabilities, caption_val_probabilities, targets, min_recall=0.95):
    search_results = []
    for multimodal_weight in np.arange(0.0, 1.01, 0.05):
        caption_weight = 1.0 - multimodal_weight
        ensemble_probabilities = multimodal_weight * multimodal_val_probabilities + caption_weight * caption_val_probabilities
        best_point, mode = select_best_operating_point(ensemble_probabilities, targets, min_recall=min_recall)
        search_results.append(
            {
                "multimodal_weight": float(np.round(multimodal_weight, 2)),
                "caption_weight": float(np.round(caption_weight, 2)),
                "threshold": best_point["threshold"],
                "precision": best_point["precision"],
                "recall": best_point["recall"],
                "f1": best_point["f1"],
                "mode": mode,
            }
        )

    recall_constrained = [row for row in search_results if row["mode"] == "recall_constrained"]
    if recall_constrained:
        best = max(recall_constrained, key=lambda row: (row["precision"], row["f1"], -row["multimodal_weight"]))
    else:
        best = max(search_results, key=lambda row: (row["recall"], row["precision"], row["f1"], -row["multimodal_weight"]))
    return best, search_results


def run_ensemble_inference(test_jsonl_path, data_root, min_recall=0.95):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    caption_bundle = joblib.load(caption_model_path)
    caption_model = caption_bundle["model"]
    model_name = caption_bundle["model_name"]
    max_length = int(caption_bundle["max_length"])

    records = read_test_records(test_jsonl_path)
    test_texts = [str(row.get("text", "")) for row in records]
    test_cache_path = data_root / "caption_only_embeddings" / "test_minilm.npy"
    test_caption_embeddings = encode_texts(
        test_texts,
        model_name=model_name,
        max_length=max_length,
        device=device,
        cache_path=test_cache_path,
    )
    caption_test_probabilities = caption_model.predict_proba(np.asarray(test_caption_embeddings))[:, 1].astype(np.float32, copy=False)

    multimodal_model, checkpoint_source = load_multimodal_model(device)
    val_image_features, val_text_features, val_labels = load_split_embeddings("val")
    test_image_features, test_text_features, _ = load_split_embeddings("test")

    multimodal_val_probabilities = collect_multimodal_probabilities(
        multimodal_model,
        val_image_features,
        val_text_features,
        device=device,
    )
    multimodal_test_probabilities = collect_multimodal_probabilities(
        multimodal_model,
        test_image_features,
        test_text_features,
        device=device,
    )

    if caption_val_prob_path.exists() and caption_val_target_path.exists():
        caption_val_probabilities = np.load(caption_val_prob_path)
        caption_val_targets = np.load(caption_val_target_path)
        if len(multimodal_val_probabilities) != len(caption_val_probabilities):
            raise ValueError("Caption-only and multimodal validation predictions have different lengths.")
        if len(caption_val_targets) != len(caption_val_probabilities):
            raise ValueError("Caption-only validation targets and probabilities have different lengths.")
        best_ensemble, ensemble_search_results = choose_ensemble_weight(
            multimodal_val_probabilities,
            caption_val_probabilities,
            caption_val_targets,
            min_recall=min_recall,
        )
        multimodal_weight = best_ensemble["multimodal_weight"]
        caption_weight = best_ensemble["caption_weight"]
        ensemble_threshold = best_ensemble["threshold"]
        ensemble_mode = best_ensemble["mode"]
    else:
        multimodal_weight = 0.5
        caption_weight = 0.5
        ensemble_threshold = 0.5
        ensemble_mode = "default_equal_weight"
        ensemble_search_results = []

    ensemble_probabilities = (
        multimodal_weight * multimodal_test_probabilities + caption_weight * caption_test_probabilities
    ).astype(np.float32, copy=False)
    final_labels = (ensemble_probabilities >= ensemble_threshold).astype(np.int64)

    output_records = []
    for row, multimodal_prob, caption_prob, ensemble_prob, label in zip(
        records,
        multimodal_test_probabilities,
        caption_test_probabilities,
        ensemble_probabilities,
        final_labels,
    ):
        output_records.append(
            {
                "id": row.get("id"),
                "img": row.get("img"),
                "text": row.get("text", ""),
                "multimodal_probability": float(multimodal_prob),
                "caption_probability": float(caption_prob),
                "ensemble_probability": float(ensemble_prob),
                "label": int(label),
            }
        )

    with open(output_jsonl_path, "w", encoding="utf-8") as f:
        for row in output_records:
            f.write(json.dumps(row) + "\n")

    np.save(output_npy_path, ensemble_probabilities)
    np.savez(
        ensemble_config_path,
        multimodal_weight=np.array([multimodal_weight], dtype=np.float32),
        caption_weight=np.array([caption_weight], dtype=np.float32),
        ensemble_threshold=np.array([ensemble_threshold], dtype=np.float32),
    )

    return {
        "records": output_records,
        "ensemble_probabilities": ensemble_probabilities,
        "caption_probabilities": caption_test_probabilities,
        "multimodal_probabilities": multimodal_test_probabilities,
        "multimodal_val_probabilities": multimodal_val_probabilities,
        "final_labels": final_labels,
        "multimodal_weight": multimodal_weight,
        "caption_weight": caption_weight,
        "ensemble_threshold": ensemble_threshold,
        "ensemble_mode": ensemble_mode,
        "ensemble_search_results": ensemble_search_results,
        "checkpoint_source": checkpoint_source,
    }


ensemble_results = run_ensemble_inference(test_jsonl_path, data_root)

print(f"Used multimodal checkpoint: {ensemble_results['checkpoint_source']}")
print(f"Saved ensemble predictions to: {output_jsonl_path}")
print(f"Saved ensemble probabilities to: {output_npy_path}")
print(f"Saved ensemble config to: {ensemble_config_path}")
print(f"Test rows processed: {len(ensemble_results['records'])}")
print(f"Predicted harmful count: {int(ensemble_results['final_labels'].sum())}")
print(f"Predicted safe count: {int((ensemble_results['final_labels'] == 0).sum())}")
print(f"First prediction: {ensemble_results['records'][0] if ensemble_results['records'] else 'N/A'}")


RuntimeError: AdvancedMultimodalClassifier is not defined. Run the multimodal model definition cell before the ensemble cell.